In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Optional: For pretty scientific plots
try:
    import scienceplots
    plt.style.use(["science", "grid", "high-vis", "no-latex"])
except ImportError:
    plt.style.use("default")

Imports and Setup

In [5]:
import os
print(os.listdir('/content/drive/MyDrive/AML_Semantic_Segmentation/PIDNet/datasets/'))

['__pycache__', 'base_dataset.py', 'readme.txt', 'camvid.py', '__init__.py', 'cityscapes.py', 'loveda.py']


Instantiate the Dataset

In [6]:
# Set these paths to your actual data locations
root = '/content/drive/MyDrive/AML_Semantic_Segmentation/data/LoveDA/Train/Urban/images_png'
list_path = '/content/drive/MyDrive/AML_Semantic_Segmentation/PIDNet/data/list/loveda/train.lst'

train_dataset = LoveDA(
    root=root,
    list_path=list_path,
    num_classes=7,
    multi_scale=False,
    flip=False,
    ignore_label=255,
    base_size=1024,
    crop_size=(1024, 1024)
)

print(f"Train Dataset length: {len(train_dataset)}")

# Set these paths to your actual data locations
root = '/content/drive/MyDrive/AML_Semantic_Segmentation/data/LoveDA/Val/Urban/images_png'
list_path = '/content/drive/MyDrive/AML_Semantic_Segmentation/PIDNet/data/list/loveda/val.lst'

val_dataset = LoveDA(
    root=root,
    list_path=list_path,
    num_classes=7,
    multi_scale=False,
    flip=False,
    ignore_label=255,
    base_size=1024,
    crop_size=(1024, 1024)
)

print(f"Val Dataset length: {len(val_dataset)}")

NameError: name 'LoveDA' is not defined

Visualize a Few Samples and Print Label Ranges

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

sys.path.append('/content/drive/MyDrive/AML_Semantic_Segmentation/PIDNet/tools')  # so you can import datasets
sys.path.append('/content/drive/MyDrive/AML_Semantic_Segmentation/PIDNet/')
from visualization import decode_segmap

# Use your dataset's mean and std (from config or dataset)
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

num_samples = 2
all_labels = []

for idx in range(num_samples):
    image, label, edge, size, name = train_dataset[idx]
    # image: torch.Tensor (C, H, W) normalized
    # label: np.ndarray or torch.Tensor (H, W)

    # Unnormalize image for visualization
    if isinstance(image, torch.Tensor):
        img_np = image.numpy().transpose(1,2,0)  # (H, W, C)
    else:
        img_np = image.transpose(1,2,0)
    img_np = (img_np * std + mean)
    img_np = np.clip(img_np, 0, 1)

    # Visualize the label mask with color
    label_vis = decode_segmap(label, ignore_index=-1)  # or whatever your ignore index is

    plt.figure(figsize=(12,6))
    plt.subplot(1,2,1)
    plt.imshow(img_np)
    plt.title(f"Original Image: {name}")
    plt.axis('off')

    plt.subplot(1,2,2)
    plt.imshow(label_vis)
    plt.title(f"Label visualization for {name}")
    plt.axis('off')
    plt.show()

    # See unique label values
    print("Unique label values in this mask:", np.unique(label))
    all_labels.append(label)

# Flatten all labels and print global min/max/unique
all_labels_flat = np.concatenate([l.flatten() for l in all_labels])
print(f"\nGlobal label value range in {num_samples} samples: {all_labels_flat.min()} to {all_labels_flat.max()}")
print(f"Unique label values in {num_samples} samples: {np.unique(all_labels_flat)}")


Histogram of Label Distribution

In [ ]:
plt.figure(figsize=(8,4))
plt.hist(all_labels_flat, bins=np.arange(-1,8)-0.5, rwidth=0.8)
plt.title("Label Value Distribution in Sampled Masks")
plt.xlabel("Label Value")
plt.ylabel("Pixel Count")
plt.xticks(np.arange(0, 7))
plt.show()

In [ ]:
import numpy as np
from tqdm import tqdm
from loveda import LoveDA  # adjust import if needed

# Instantiate your dataset (adjust paths/config as needed)
dataset = LoveDA(
    root='data/',  # or your actual root
    list_path='/content/drive/MyDrive/AML_Semantic_Segmentation/PIDNet/data/list/loveda/val.lst',  # or your actual list
    num_classes=7,
    multi_scale=False,
    flip=False,
    ignore_label=255,
    base_size=720,
    crop_size=(512, 512)
)

all_labels = []

for i in tqdm(range(len(dataset)), desc="Scanning dataset"):
    _, label, _, _, _ = dataset[i]
    all_labels.append(label.flatten())

all_labels = np.concatenate(all_labels)
unique_labels = np.unique(all_labels)

print(f"Unique label values in the dataset: {unique_labels}")
print(f"Label value range: min={unique_labels.min()}, max={unique_labels.max()}")

import collections
counts = collections.Counter(all_labels)
print("Counts per label:", dict(counts))

# RUN

Install Dependencies

In [ ]:
!pip install tensorboardX thop albumentations tqdm pyyaml yacs

Generate dataset list files

In [ ]:
%cd /content/drive/MyDrive/AML_Semantic_Segmentation/PIDNet

!python -m datasets.loveda

In [ ]:
import sys

sys.path.append('/content/drive/MyDrive/AML_Semantic_Segmentation/PIDNet/')

sys.path.append('/content/drive/MyDrive/AML_Semantic_Segmentation/PIDNet/datasets/')

sys.path.append('/content/drive/MyDrive/AML_Semantic_Segmentation/PIDNet/tools/')

Train PIDNet-S on Imagenet

In [ ]:
 se

In [ ]:
%cd /content/drive/MyDrive/AML_Semantic_Segmentation/PIDNet
!python ./tools/train.py --cfg ./configs/loveda/pidnet_loveda_urban.yaml

Evaluate on LoveDA-urban

In [ ]:
!python eval.py --cfg pidnet_loveda_urban.yaml

Visualize preidctions

In [ ]:
!python visualization.py --config pidnet_loveda_urban.yaml --checkpoint output/best.pt --split val --output_dir visualizations --num_samples 10

Show some visualizations

In [ ]:
import matplotlib.pyplot as plt
import glob
from PIL import Image

vis_dir = 'visualizations'
imgs = sorted(glob.glob(f"{vis_dir}/*.png"))
for img_path in imgs[:5]:
    img = Image.open(img_path)
    plt.figure(figsize=(10,5))
    plt.imshow(img)
    plt.axis('off')
    plt.show()